# 12 Class 2 vs 3: is there a signal?

Segment features (notebook 10), Kaggle classes 2 and 3 only, 160 development segments, twin-grouped folds from notebook 08. The hold-out is not used. Labels are Kaggle's.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.config import PROCESSED, RANDOM_STATE
from src.cv import load_fold_table
from src.evaluate import cross_validate_segments
from src.features import FEATURES, SIZE_FEATURES, SHAPE_FEATURES

feat = pd.read_csv(PROCESSED / "features_dev.csv")
table = load_fold_table("cv_folds_allclass.csv")
data = feat[feat["y"].isin([2, 3])].reset_index(drop=True)          # Kaggle classes 2 and 3, development set only
assert set(data["segment"]) <= set(table["segment"])

models = {
    "chance": lambda: DummyClassifier(strategy="prior"),
    "logistic regression": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "random forest": lambda: RandomForestClassifier(200, random_state=RANDOM_STATE, n_jobs=-1),
}

fold0 = data["segment"].map(table.set_index("segment")["repeat_0"])
c = pd.crosstab(fold0, data["y"])
print("segments:", len(data), data["y"].value_counts().sort_index().to_dict(), "| twin groups:", data["group"].nunique())
print("segments per fold:", sorted(c.sum(axis=1).unique()), "| class 2 per fold:", sorted(c[2].unique()), "| class 3 per fold:", sorted(c[3].unique()))

segments: 160 {2: 80, 3: 80} | twin groups: 76
segments per fold: [np.int64(15), np.int64(16), np.int64(17)] | class 2 per fold: [np.int64(8)] | class 3 per fold: [np.int64(7), np.int64(8), np.int64(9)]


## Grid: feature set x model

In [2]:
feature_sets = {"all 23 features": FEATURES, "size only (8)": SIZE_FEATURES, "shape only (15)": SHAPE_FEATURES}
results = {}
for fname, cols in feature_sets.items():
    for mname, make_model in models.items():
        if mname == "chance" and fname != "all 23 features":
            continue
        results[(fname, mname)] = cross_validate_segments(make_model, data, cols, table, positive=2)

summary = pd.DataFrame({k: v.drop(columns="repeat").mean() for k, v in results.items()}).T
summary["auc_sd"] = pd.Series({k: v["auc_fold"].std() for k, v in results.items()})
summary[["auc_fold", "auc_sd", "pr_auc_fold", "sensitivity", "specificity", "precision", "f1", "balanced_accuracy"]].round(3)

auc_fold  auc_sd  pr_auc_fold  \
all 23 features chance                  0.500   0.000        0.500   
                logistic regression     0.708   0.025        0.797   
                random forest           0.770   0.023        0.806   
size only (8)   logistic regression     0.664   0.012        0.748   
                random forest           0.738   0.025        0.780   
shape only (15) logistic regression     0.749   0.018        0.821   
                random forest           0.779   0.024        0.808   

                                     sensitivity  specificity  precision  \
all 23 features chance                     0.100        0.888      0.471   
                logistic regression        0.610        0.732      0.695   
                random forest              0.668        0.732      0.714   
size only (8)   logistic regression        0.560        0.688      0.642   
                random forest              0.610        0.753      0.711   
shape only (15) logistic regression        0.640        0.768      0.735   
                random forest              0.695        0.765      0.747   

                                        f1  balanced_accuracy  
all 23 features chance               0.165              0.494  
                logistic regression  0.650              0.671  
                random forest        0.690              0.700  
size only (8)   logistic regression  0.598              0.624  
                random forest        0.657              0.681  
shape only (15) logistic regression  0.684              0.704  
                random forest        0.720              0.730

## Shuffled-label ranges

Shuffle the labels between whole twin groups (30 draws, class sizes kept within 4 segments), run the same CV once per draw. This is what a model with no real signal scores.

In [3]:
group_class = data.groupby("group")["y"].agg(lambda s: s.value_counts().idxmax())
group_size = data.groupby("group").size()
n_pos = (data["y"] == 2).sum()

def null_range(make_model, cols, n_draws=30):
    chance, s = [], 0
    while len(chance) < n_draws:
        rng = np.random.default_rng(100 + s)
        s += 1
        shuffled = pd.Series(rng.permutation(group_class.values), index=group_class.index)    # labels shuffled between groups
        if abs(group_size[shuffled == 2].sum() - n_pos) > 4:
            continue
        d = data.copy()
        d["y"] = d["group"].map(shuffled)
        chance.append(cross_validate_segments(make_model, d, cols, table, n_repeats=1, positive=2)["auc_fold"].iloc[0])
    return np.array(chance)

for mname in ["logistic regression", "random forest"]:
    for fname in ["all 23 features", "shape only (15)"]:
        c = null_range(models[mname], feature_sets[fname])
        print(f"{mname:20s} {fname:16s} shuffled-label 5th-95th: {np.round(np.percentile(c, [5, 95]), 2)} | mean {c.mean():.3f}")

logistic regression  all 23 features  shuffled-label 5th-95th: [0.32 0.64] | mean 0.488


logistic regression  shape only (15)  shuffled-label 5th-95th: [0.32 0.62] | mean 0.481


random forest        all 23 features  shuffled-label 5th-95th: [0.32 0.65] | mean 0.512


random forest        shape only (15)  shuffled-label 5th-95th: [0.35 0.66] | mean 0.499


## Findings

| AUC | logistic | forest |
|---|---|---|
| all 23 | 0.708 | 0.770 |
| size only | 0.664 | 0.738 |
| shape only | 0.749 | 0.779 |

Chance is 0.500. Shuffled-label ranges (5th to 95th percentile): logistic 0.32 to 0.64 (all 23) and 0.32 to 0.62 (shape); forest 0.32 to 0.65 (all 23) and 0.35 to 0.66 (shape).

- Class 2 vs 3 is above chance on segment features, unlike the chunk-level models in 07.
- Shape features do as well as or better than all 23, so this is not just signal size.
- Shape-only logistic (0.749) and both forests are above the top of their shuffled ranges. Logistic on all 23 (0.708) is only just above its range (0.64). It is a modest signal, not a strong one.

## Caveats

- No patient ID. A model may be recognising a person or site it has already seen, and this cannot be tested here.
- Twin grouping may miss weaker similarity, so these scores are upper bounds.
- 160 segments is small. Differences under 0.03 are ties.
- Nothing here says anything about tumours. These are Kaggle's labels.